# 02. Limpieza de datos

Esta notebook documenta la etapa de limpieza del dataset survey de Stack Overflow. El objetivo es transformar la fuente adquirida en la notebook anterior en una version consistente, trazable y apta para EDA, visualizaciones y dashboarding.

## Alcance

- cargar el dataset survey base
- perfilar faltantes y columnas problematicas
- estandarizar valores categoricos con inconsistencias visibles
- imputar faltantes con reglas diferenciadas por tipo de variable
- generar una copia reducida para analisis posterior
- exportar salidas limpias y diagnosticos


## Subpaso 1. Carga del dataset y rutas de salida

**Proposito:** iniciar la etapa de limpieza con una fuente reproducible y una carpeta de exportacion dedicada.

**Resultado esperado:** acceso al archivo `data/survey_data_updated.csv` y a la carpeta `data/cleaned_outputs`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "cleaned_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SURVEY_CSV_PATH = DATA_DIR / "survey_data_updated.csv"
SURVEY_CSV_PATH, OUTPUT_DIR


## Subpaso 2. Perfil inicial de calidad

**Proposito:** dimensionar el problema de limpieza antes de modificar el dataset.

**Resultado:** el snapshot contiene `18,845` filas, `114` columnas y `465,762` valores faltantes en total.

**Hallazgos:** las columnas mas afectadas corresponden sobre todo a preguntas opcionales de AI, embedded y herramientas, mientras que `ConvertedCompYearly` tambien aparece con una cantidad importante de faltantes.


In [ ]:
survey_raw = pd.read_csv(SURVEY_CSV_PATH)

missing_before = survey_raw.isna().sum().sort_values(ascending=False)
profile_before = pd.DataFrame(
    {
        "dtype": survey_raw.dtypes.astype(str),
        "missing_values": survey_raw.isna().sum(),
        "non_null_count": survey_raw.count(),
    }
)

print("Shape:", survey_raw.shape)
print("Total missing values:", int(survey_raw.isna().sum().sum()))
missing_before.head(15)


## Subpaso 3. Estandarizacion de categorias inconsistentes

**Proposito:** corregir valores textuales que afectan legibilidad y consistencia de categorias.

**Resultado:** se normalizan algunas etiquetas de `Country` y `EdLevel` para facilitar analisis y visualizaciones posteriores.

**Hallazgo breve:** varias inconsistencias visibles provienen de nombres demasiado largos o problemas de codificacion de caracteres en niveles educativos.


In [ ]:
survey_clean = survey_raw.copy()

country_mapping = {
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
}

edlevel_mapping = {
    "Bachelor�s degree (B.A., B.S., B.Eng., etc.)": "Bachelors degree",
    "Master�s degree (M.A., M.S., M.Eng., MBA, etc.)": "Masters degree",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional degree",
    "Associate degree (A.A., A.S., etc.)": "Associate degree",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary school",
    "Primary/elementary school": "Primary school",
}

survey_clean["Country"] = survey_clean["Country"].replace(country_mapping)
survey_clean["EdLevel"] = survey_clean["EdLevel"].replace(edlevel_mapping)

survey_clean[["Country", "EdLevel"]].head()


## Subpaso 4. Definicion de reglas de imputacion

**Proposito:** separar el tratamiento de faltantes segun el tipo de variable y el uso analitico esperado.

**Resultado:** se definen tres grupos principales: variables numericas imputadas con mediana, variables categoricas estructuradas imputadas con moda y variables libres o multiseleccion imputadas con `Not specified`.

**Hallazgo metodologico:** `ConvertedCompYearly` tambien se trata en esta notebook, porque en este snapshot aun conserva muchos faltantes y dejarla incompleta dificultaria EDA, visualizaciones y comparaciones posteriores.


In [ ]:
numeric_impute_cols = [
    "CompTotal",
    "WorkExp",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5",
    "JobSatPoints_6",
    "JobSatPoints_7",
    "JobSatPoints_8",
    "JobSatPoints_9",
    "JobSatPoints_10",
    "JobSatPoints_11",
    "JobSat",
    "ConvertedCompYearly",
]

mode_impute_cols = [
    "RemoteWork",
    "EdLevel",
    "OrgSize",
    "PurchaseInfluence",
    "BuildvsBuy",
    "SOVisitFreq",
    "SOAccount",
    "SOPartFreq",
    "SOComm",
    "AISelect",
    "AISent",
    "AIAcc",
    "AIComplex",
    "AIThreat",
    "TBranch",
    "ICorPM",
    "Knowledge_1",
    "Knowledge_2",
    "Knowledge_3",
    "Knowledge_4",
    "Knowledge_5",
    "Knowledge_6",
    "Knowledge_7",
    "Knowledge_8",
    "Knowledge_9",
    "Frequency_1",
    "Frequency_2",
    "Frequency_3",
    "TimeSearching",
    "TimeAnswering",
    "ProfessionalCloud",
    "ProfessionalQuestion",
    "Industry",
    "SurveyLength",
    "SurveyEase",
]

not_specified_cols = [
    "LearnCode",
    "LearnCodeOnline",
    "TechDoc",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "BuyNewTool",
    "TechEndorse",
    "Country",
    "Currency",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DatabaseAdmired",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "PlatformAdmired",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "WebframeAdmired",
    "EmbeddedHaveWorkedWith",
    "EmbeddedWantToWorkWith",
    "EmbeddedAdmired",
    "MiscTechHaveWorkedWith",
    "MiscTechWantToWorkWith",
    "MiscTechAdmired",
    "ToolsTechHaveWorkedWith",
    "ToolsTechWantToWorkWith",
    "ToolsTechAdmired",
    "NEWCollabToolsHaveWorkedWith",
    "NEWCollabToolsWantToWorkWith",
    "NEWCollabToolsAdmired",
    "OpSysPersonal use",
    "OpSysProfessional use",
    "OfficeStackAsyncHaveWorkedWith",
    "OfficeStackAsyncWantToWorkWith",
    "OfficeStackAsyncAdmired",
    "OfficeStackSyncHaveWorkedWith",
    "OfficeStackSyncWantToWorkWith",
    "OfficeStackSyncAdmired",
    "AISearchDevHaveWorkedWith",
    "AISearchDevWantToWorkWith",
    "AISearchDevAdmired",
    "NEWSOSites",
    "SOHow",
    "AIBen",
    "AIToolCurrently Using",
    "AIToolInterested in Using",
    "AIToolNot interested in Using",
    "AINextMuch more integrated",
    "AINextNo change",
    "AINextMore integrated",
    "AINextLess integrated",
    "AINextMuch less integrated",
    "AIEthics",
    "AIChallenges",
    "Frustration",
    "ProfessionalTech",
]

len(numeric_impute_cols), len(mode_impute_cols), len(not_specified_cols)


## Subpaso 5. Imputacion y limpieza principal

**Proposito:** aplicar las reglas de imputacion sobre el dataset base.

**Resultado:** se completa el tratamiento de faltantes y se obtiene una version limpia con `0` valores nulos.

**Hallazgos:** `RemoteWork` se completa con su categoria modal y `ConvertedCompYearly` se imputa con mediana para preservar la mayoria de registros en analisis posteriores.


In [ ]:
imputation_log = []

for column in numeric_impute_cols:
    if column in survey_clean.columns:
        missing_before_col = int(survey_clean[column].isna().sum())
        median_value = survey_clean[column].median()
        survey_clean[column] = survey_clean[column].fillna(median_value)
        imputation_log.append(
            {
                "column_name": column,
                "strategy": "median",
                "missing_before": missing_before_col,
                "fill_value": median_value,
            }
        )

for column in mode_impute_cols:
    if column in survey_clean.columns:
        missing_before_col = int(survey_clean[column].isna().sum())
        mode_series = survey_clean[column].mode(dropna=True)
        fill_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
        survey_clean[column] = survey_clean[column].fillna(fill_value)
        imputation_log.append(
            {
                "column_name": column,
                "strategy": "mode",
                "missing_before": missing_before_col,
                "fill_value": fill_value,
            }
        )

for column in not_specified_cols:
    if column in survey_clean.columns:
        missing_before_col = int(survey_clean[column].isna().sum())
        survey_clean[column] = survey_clean[column].fillna("Not specified")
        imputation_log.append(
            {
                "column_name": column,
                "strategy": "not_specified",
                "missing_before": missing_before_col,
                "fill_value": "Not specified",
            }
        )

missing_after_cleaning = int(survey_clean.isna().sum().sum())
imputation_log_df = pd.DataFrame(imputation_log)

print("Total missing values after cleaning:", missing_after_cleaning)
imputation_log_df.head(12)


## Subpaso 6. Derivacion de campos de compensacion

**Proposito:** añadir versiones normalizadas de `ConvertedCompYearly` para apoyar comparaciones y analisis numericos posteriores.

**Resultado:** se generan `ConvertedCompYearly_MinMax` y `ConvertedCompYearly_Zscore`.

**Hallazgo breve:** la mediana usada para imputar `ConvertedCompYearly` en este snapshot es `65,857.5`, lo que ayuda a mantener una referencia central robusta en una variable sesgada.


In [ ]:
comp_series = survey_clean["ConvertedCompYearly"]
comp_min = comp_series.min()
comp_max = comp_series.max()
comp_mean = comp_series.mean()
comp_std = comp_series.std(ddof=0)

survey_clean = survey_clean.assign(
    ConvertedCompYearly_MinMax=((comp_series - comp_min) / (comp_max - comp_min)) if comp_max != comp_min else 0.0,
    ConvertedCompYearly_Zscore=((comp_series - comp_mean) / comp_std) if comp_std else 0.0,
).copy()

survey_clean[["ConvertedCompYearly", "ConvertedCompYearly_MinMax", "ConvertedCompYearly_Zscore"]].head()


## Subpaso 7. Creacion de una copia reducida para analisis

**Proposito:** simplificar el dataset para usos posteriores y conservar solo las columnas que aportan directamente al relato analitico del proyecto: demografia, experiencia, salario, satisfaccion y tecnologias.

**Resultado:** la copia reducida pasa de ser un recorte minimo a un dataset enfocado, eliminando bloques perifericos como preguntas detalladas de AI, Stack Overflow usage, collaboration tools, office stack, embedded, misc tech, autoevaluaciones y metadatos de encuesta.

**Hallazgos:** la version limpia completa queda con `116` columnas, mientras que la copia reducida queda con `32` columnas y mantiene la misma cantidad de filas. Se conserva `CompTotal` por compatibilidad con las visualizaciones actuales, aunque `ConvertedCompYearly` sigue siendo la medida salarial principal.


In [ ]:
focused_reduced_columns = [
    "ResponseId",
    "MainBranch",
    "Age",
    "Employment",
    "RemoteWork",
    "EdLevel",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "OrgSize",
    "Country",
    "Currency",
    "CompTotal",
    "ConvertedCompYearly",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
    "WorkExp",
    "Industry",
    "JobSat",
    "JobSatPoints_6",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DatabaseAdmired",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "PlatformAdmired",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "WebframeAdmired",
]

focused_reduced_columns = [column for column in focused_reduced_columns if column in survey_clean.columns]
drop_columns_for_reduced = [column for column in survey_clean.columns if column not in focused_reduced_columns]

survey_clean_reduced = survey_clean[focused_reduced_columns].copy()
reduced_columns_manifest = pd.DataFrame(
    {
        "kept_column": pd.Series(focused_reduced_columns),
    }
)
removed_columns_manifest = pd.DataFrame(
    {
        "removed_column": pd.Series(drop_columns_for_reduced),
    }
)

print("Cleaned shape:", survey_clean.shape)
print("Reduced shape:", survey_clean_reduced.shape)
print("Removed columns count:", len(drop_columns_for_reduced))
removed_columns_manifest.head(20)


## Subpaso 8. Exportacion de salidas y diagnosticos

**Proposito:** persistir datasets limpios y tablas de control para reutilizarlos en las etapas siguientes.

**Resultado:** se exportan una version limpia completa, una version reducida y archivos de diagnostico sobre faltantes e imputacion.

**Hallazgo operativo:** centralizar estas salidas evita repetir el mismo wrangling en cada notebook y deja una trazabilidad clara del estado del dataset despues de limpieza.


In [ ]:
profile_after = pd.DataFrame(
    {
        "dtype": survey_clean.dtypes.astype(str),
        "missing_values": survey_clean.isna().sum(),
        "non_null_count": survey_clean.count(),
    }
)

missing_comparison = pd.DataFrame(
    {
        "missing_before": survey_raw.isna().sum(),
        "missing_after": survey_clean.isna().sum(),
    }
).sort_values("missing_before", ascending=False)

cleaned_csv_path = OUTPUT_DIR / "survey_data_cleaned.csv"
reduced_csv_path = OUTPUT_DIR / "survey_data_cleaned_reduced.csv"
profile_before_csv_path = OUTPUT_DIR / "survey_profile_before_cleaning.csv"
profile_after_csv_path = OUTPUT_DIR / "survey_profile_after_cleaning.csv"
missing_comparison_csv_path = OUTPUT_DIR / "survey_missingness_comparison.csv"
imputation_log_csv_path = OUTPUT_DIR / "survey_imputation_log.csv"
reduced_kept_columns_csv_path = OUTPUT_DIR / "survey_reduced_kept_columns.csv"
reduced_removed_columns_csv_path = OUTPUT_DIR / "survey_reduced_removed_columns.csv"

survey_clean.to_csv(cleaned_csv_path, index=False)
survey_clean_reduced.to_csv(reduced_csv_path, index=False)
profile_before.reset_index(names="column_name").to_csv(profile_before_csv_path, index=False)
profile_after.reset_index(names="column_name").to_csv(profile_after_csv_path, index=False)
missing_comparison.reset_index(names="column_name").to_csv(missing_comparison_csv_path, index=False)
imputation_log_df.to_csv(imputation_log_csv_path, index=False)
reduced_columns_manifest.to_csv(reduced_kept_columns_csv_path, index=False)
removed_columns_manifest.to_csv(reduced_removed_columns_csv_path, index=False)

print("Saved:")
print(cleaned_csv_path)
print(reduced_csv_path)
print(profile_before_csv_path)
print(profile_after_csv_path)
print(missing_comparison_csv_path)
print(imputation_log_csv_path)
print(reduced_kept_columns_csv_path)
print(reduced_removed_columns_csv_path)


## Cierre de la etapa

La limpieza deja dos salidas reutilizables: una version completa y una copia reducida del dataset survey. La siguiente notebook puede partir desde estas tablas para concentrarse en EDA y estadisticas, en lugar de repetir tratamiento de faltantes y estandarizacion.
